In [1]:
#prediction test 
import joblib
import pandas as pd

MODEL_PATH = "grade_predictor_rf.joblib"
model = joblib.load(MODEL_PATH)

def predict_grade(student_data: dict) -> float:
    df = pd.DataFrame([student_data])
    pred = model.predict(df)[0]
    return float(round(pred, 2))

def recommend_changes(student_data: dict, target_grade: float, raw_dataset_path_mat="student-mat.csv", raw_dataset_path_por="student-por.csv"):
    # Load raw data to build "what does success look like" stats
    mat = pd.read_csv(raw_dataset_path_mat, sep=";")
    por = pd.read_csv(raw_dataset_path_por, sep=";")
    mat["subject"] = "math"
    por["subject"] = "portuguese"
    df = pd.concat([mat, por], ignore_index=True)

    # Same cleaning as model
    df["absences"] = df["absences"].clip(upper=30)
    df = df.drop(columns=["G1", "G2"])

    # Filter to same subject as the user (better recommendations)
    subj = student_data.get("subject", None)
    if subj in ["math", "portuguese"]:
        df_sub = df[df["subject"] == subj].copy()
    else:
        df_sub = df.copy()

    # Students who met/exceeded target
    winners = df_sub[df_sub["G3"] >= target_grade].copy()
    if len(winners) < 30:
        # if too few, loosen the threshold slightly
        winners = df_sub[df_sub["G3"] >= (target_grade - 1)].copy()

    # Compute typical values among "winners"
    # (these are the features we can reasonably recommend changes for)
    actionable = ["studytime", "absences", "goout", "Walc", "Dalc", "freetime"]
    typical = winners[actionable].median(numeric_only=True)

    # Build suggestions by comparing user to typical
    suggestions = []

    # studytime: higher is usually better
    if student_data["studytime"] < typical["studytime"]:
        suggestions.append(f"Increase studytime from {student_data['studytime']} → ~{int(typical['studytime'])} (students hitting {target_grade}+ usually study more).")

    # absences: lower is usually better
    if student_data["absences"] > typical["absences"]:
        suggestions.append(f"Reduce absences from {student_data['absences']} → ~{int(typical['absences'])} (top students miss fewer classes).")

    # going out / alcohol: lower often correlates with higher grades
    if student_data["goout"] > typical["goout"]:
        suggestions.append(f"Cut down 'goout' from {student_data['goout']} → ~{int(typical['goout'])} (more consistent routine helps).")

    if student_data["Walc"] > typical["Walc"]:
        suggestions.append(f"Lower weekend alcohol (Walc) from {student_data['Walc']} → ~{int(typical['Walc'])} (higher performers tend to be lower).")

    if student_data["Dalc"] > typical["Dalc"]:
        suggestions.append(f"Lower weekday alcohol (Dalc) from {student_data['Dalc']} → ~{int(typical['Dalc'])} (helps consistency).")

    # freetime is ambiguous; don’t over-prescribe, just note extremes
    if student_data["freetime"] > typical["freetime"] + 1:
        suggestions.append(f"Consider structuring some free time (freetime {student_data['freetime']}) toward study blocks.")

    # failures: if >0 it’s a major flag; suggest support
    if student_data["failures"] > 0:
        suggestions.append("Because you have past failures > 0, prioritize extra support (teacher help, tutoring, structured revision plan).")

    if not suggestions:
        suggestions.append("Your habits already look similar to students who hit that target. Keep consistency and avoid increasing absences.")

    return typical.to_dict(), suggestions

if __name__ == "__main__":
    example_student = {
        "school": "GP",
        "sex": "F",
        "age": 17,
        "address": "U",
        "famsize": "GT3",
        "Pstatus": "T",
        "Medu": 3,
        "Fedu": 3,
        "Mjob": "other",
        "Fjob": "other",
        "reason": "course",
        "guardian": "mother",
        "traveltime": 1,
        "studytime": 3,
        "failures": 0,
        "schoolsup": "no",
        "famsup": "yes",
        "paid": "no",
        "activities": "yes",
        "nursery": "yes",
        "higher": "yes",
        "internet": "yes",
        "romantic": "no",
        "famrel": 4,
        "freetime": 3,
        "goout": 2,
        "Dalc": 1,
        "Walc": 2,
        "health": 4,
        "absences": 3,
        "subject": "math"
    }

    pred = predict_grade(example_student)
    print("Predicted grade:", pred)

    target = 15
    typical, suggestions = recommend_changes(example_student, target_grade=target)
    print(f"\nTypical habits for {target}+ (approx medians):", typical)
    print("\nSuggestions:")
    for s in suggestions:
        print("-", s)

Predicted grade: 12.81

Typical habits for 15+ (approx medians): {'studytime': 2.0, 'absences': 2.0, 'goout': 3.0, 'Walc': 1.0, 'Dalc': 1.0, 'freetime': 3.0}

Suggestions:
- Reduce absences from 3 → ~2 (top students miss fewer classes).
- Lower weekend alcohol (Walc) from 2 → ~1 (higher performers tend to be lower).


In [2]:
# If ipywidgets isn't installed/enabled, uncomment the next line:
# !pip install ipywidgets

import pandas as pd
import joblib

from IPython.display import display, Markdown
import ipywidgets as widgets
from ipywidgets import VBox, HBox, GridBox, Layout

# Load trained model
model = joblib.load("grade_predictor_rf.joblib")

# Load datasets once (used for recommendation stats)
mat = pd.read_csv("student-mat.csv", sep=";")
por = pd.read_csv("student-por.csv", sep=";")
mat["subject"] = "math"
por["subject"] = "portuguese"
raw_all = pd.concat([mat, por], ignore_index=True)

# Clean the same way as training (early-term)
raw_all["absences"] = raw_all["absences"].clip(upper=30)
raw_all = raw_all.drop(columns=["G1", "G2"])

In [3]:
def predict_grade(student_data: dict) -> float:
    df_one = pd.DataFrame([student_data])
    pred = model.predict(df_one)[0]
    return round(float(pred), 2)

def recommend_changes(student_data: dict, target_grade: float, df_all: pd.DataFrame):
    # Compare against same subject if possible
    subj = student_data.get("subject", None)
    if subj in ["math", "portuguese"]:
        df = df_all[df_all["subject"] == subj].copy()
    else:
        df = df_all.copy()

    winners = df[df["G3"] >= target_grade].copy()
    if len(winners) < 30:
        winners = df[df["G3"] >= (target_grade - 1)].copy()

    actionable = ["studytime", "absences", "goout", "Walc", "Dalc", "freetime", "failures"]
    typical = winners[actionable].median(numeric_only=True)

    suggestions = []

    if student_data["failures"] > 0:
        suggestions.append("You have past failures > 0 → prioritize extra support (tutoring/teacher help + structured revision plan).")

    if student_data["studytime"] < typical["studytime"]:
        suggestions.append(f"Increase studytime: {student_data['studytime']} → ~{int(typical['studytime'])}")

    if student_data["absences"] > typical["absences"]:
        suggestions.append(f"Reduce absences: {student_data['absences']} → ~{int(typical['absences'])}")

    if student_data["goout"] > typical["goout"]:
        suggestions.append(f"Reduce going out (goout): {student_data['goout']} → ~{int(typical['goout'])}")

    if student_data["Walc"] > typical["Walc"]:
        suggestions.append(f"Lower weekend alcohol (Walc): {student_data['Walc']} → ~{int(typical['Walc'])}")

    if student_data["Dalc"] > typical["Dalc"]:
        suggestions.append(f"Lower weekday alcohol (Dalc): {student_data['Dalc']} → ~{int(typical['Dalc'])}")

    if not suggestions:
        suggestions.append("Your profile already looks similar to students reaching this target. Keep consistency and avoid more absences.")

    return typical.to_dict(), suggestions

In [4]:
# =========================
# CELL 3 (FINAL - SELF CONTAINED)
# Full UI + explanations + visual "You vs Typical"
# Requires from earlier cells:
#   - raw_all
#   - predict_grade(student_dict)
#   - recommend_changes(student_dict, target_grade, raw_all)
# =========================

from IPython.display import display, Markdown, clear_output
import ipywidgets as widgets
from ipywidgets import VBox, HBox
import pandas as pd

# -----------------------------
# Explainers (self-contained)
# -----------------------------
VAR_INFO = {
    "studytime": {
        "label": "Weekly study time",
        "explain": "How many hours you study per week (binned).",
        "map": {1: "< 2 hours", 2: "2–5 hours", 3: "5–10 hours", 4: "> 10 hours"}
    },
    "absences": {
        "label": "School absences",
        "explain": "Number of classes missed (we cap it at 30 for modeling).",
        "map": None
    },
    "goout": {
        "label": "Going out with friends",
        "explain": "How often you go out with friends (1=very low, 5=very high).",
        "map": {1:"Very low",2:"Low",3:"Medium",4:"High",5:"Very high"}
    },
    "Walc": {
        "label": "Weekend alcohol consumption",
        "explain": "Weekend alcohol use (1=very low, 5=very high).",
        "map": {1:"Very low",2:"Low",3:"Medium",4:"High",5:"Very high"}
    },
    "Dalc": {
        "label": "Weekday alcohol consumption",
        "explain": "Weekday alcohol use (1=very low, 5=very high).",
        "map": {1:"Very low",2:"Low",3:"Medium",4:"High",5:"Very high"}
    },
    "freetime": {
        "label": "Free time after school",
        "explain": "How much free time you have after school (1=very low, 5=very high).",
        "map": {1:"Very low",2:"Low",3:"Medium",4:"High",5:"Very high"}
    },
    "failures": {
        "label": "Past class failures",
        "explain": "How many times you’ve failed a class before (0–3).",
        "map": None
    }
}

ACTIONABLE_VARS = ["studytime", "absences", "goout", "Walc", "Dalc", "freetime", "failures"]

def human_value(var, val):
    info = VAR_INFO.get(var, {})
    m = info.get("map")
    if m is None:
        # show integers nicely
        try:
            if float(val).is_integer():
                return str(int(val))
        except:
            pass
        return str(val)
    try:
        v = int(round(float(val)))
    except:
        v = val
    return m.get(v, str(val))

def make_comparison_table(student_dict, typical_dict):
    rows = []
    for var in ACTIONABLE_VARS:
        if var not in typical_dict or var not in student_dict:
            continue
        rows.append({
            "Factor": VAR_INFO[var]["label"],
            "What it means": VAR_INFO[var]["explain"],
            "You": human_value(var, student_dict[var]),
            "Typical for target": human_value(var, typical_dict[var]),
        })
    return pd.DataFrame(rows)

def bar_visual(student_dict, typical_dict):
    lines = []
    for var in ACTIONABLE_VARS:
        if var not in typical_dict or var not in student_dict:
            continue

        you = float(student_dict[var])
        typ = float(typical_dict[var])

        if var == "studytime":
            maxv = 4
        elif var in ["goout", "Walc", "Dalc", "freetime"]:
            maxv = 5
        elif var == "absences":
            maxv = 30
        elif var == "failures":
            maxv = 3
        else:
            maxv = max(5, int(max(you, typ)))

        def mkbar(v):
            filled = int(round((v / maxv) * 20))
            filled = max(0, min(20, filled))
            return "█" * filled + "░" * (20 - filled)

        label = VAR_INFO[var]["label"]
        lines.append(
            f"{label}\n"
            f"  You     : {mkbar(you)}  ({human_value(var, you)})\n"
            f"  Typical : {mkbar(typ)}  ({human_value(var, typ)})"
        )

    return "\n\n".join(lines)

# -----------------------------
# WIDTH SETTINGS (fix truncation)
# -----------------------------
LABEL_WIDTH = "300px"
WIDGET_WIDTH = "520px"
common_style = {"description_width": LABEL_WIDTH}
common_layout = widgets.Layout(width=WIDGET_WIDTH)

# -----------------------------
# WIDGETS (fully spelled labels)
# -----------------------------
sex_w = widgets.Dropdown(
    options=[("Female", "F"), ("Male", "M")],
    value="F",
    description="Gender:",
    style=common_style,
    layout=common_layout
)

age_w = widgets.IntSlider(
    value=17, min=15, max=22, step=1,
    description="Age:",
    style=common_style,
    layout=common_layout
)

address_w = widgets.Dropdown(
    options=[("Urban", "U"), ("Rural", "R")],
    value="U",
    description="Home Area:",
    style=common_style,
    layout=common_layout
)

famsize_w = widgets.Dropdown(
    options=[("≤ 3 People", "LE3"), ("> 3 People", "GT3")],
    value="GT3",
    description="Family Size:",
    style=common_style,
    layout=common_layout
)

pstatus_w = widgets.Dropdown(
    options=[("Parents Together", "T"), ("Parents Apart", "A")],
    value="T",
    description="Parents Living Together:",
    style=common_style,
    layout=common_layout
)

guardian_w = widgets.Dropdown(
    options=[("Mother", "mother"), ("Father", "father"), ("Other", "other")],
    value="mother",
    description="Primary Guardian:",
    style=common_style,
    layout=common_layout
)

reason_w = widgets.Dropdown(
    options=[("Close to Home", "home"),
             ("School Reputation", "reputation"),
             ("Course Preference", "course"),
             ("Other", "other")],
    value="course",
    description="Reason for Choosing School:",
    style=common_style,
    layout=common_layout
)

medu_w = widgets.IntSlider(
    value=3, min=0, max=4, step=1,
    description="Mother Education Level (0–4):",
    style=common_style,
    layout=common_layout
)

fedu_w = widgets.IntSlider(
    value=3, min=0, max=4, step=1,
    description="Father Education Level (0–4):",
    style=common_style,
    layout=common_layout
)

job_options = [("Teacher", "teacher"), ("Health", "health"), ("Services", "services"),
               ("At Home", "at_home"), ("Other", "other")]

mjob_w = widgets.Dropdown(
    options=job_options,
    value="other",
    description="Mother Job:",
    style=common_style,
    layout=common_layout
)

fjob_w = widgets.Dropdown(
    options=job_options,
    value="other",
    description="Father Job:",
    style=common_style,
    layout=common_layout
)

traveltime_w = widgets.IntSlider(
    value=1, min=1, max=4, step=1,
    description="Travel Time to School (1–4):",
    style=common_style,
    layout=common_layout
)

studytime_w = widgets.IntSlider(
    value=2, min=1, max=4, step=1,
    description="Weekly Study Time (1–4):",
    style=common_style,
    layout=common_layout
)

failures_w = widgets.IntSlider(
    value=0, min=0, max=3, step=1,
    description="Past Class Failures (0–3):",
    style=common_style,
    layout=common_layout
)

absences_w = widgets.IntSlider(
    value=4, min=0, max=30, step=1,
    description="School Absences (0–30):",
    style=common_style,
    layout=common_layout
)

schoolsup_w = widgets.Dropdown(
    options=[("Yes", "yes"), ("No", "no")],
    value="no",
    description="Extra Educational Support (School):",
    style=common_style,
    layout=common_layout
)

famsup_w = widgets.Dropdown(
    options=[("Yes", "yes"), ("No", "no")],
    value="yes",
    description="Extra Educational Support (Family):",
    style=common_style,
    layout=common_layout
)

paid_w = widgets.Dropdown(
    options=[("Yes", "yes"), ("No", "no")],
    value="no",
    description="Extra Paid Classes (Tutoring):",
    style=common_style,
    layout=common_layout
)

activities_w = widgets.Dropdown(
    options=[("Yes", "yes"), ("No", "no")],
    value="yes",
    description="Extracurricular Activities:",
    style=common_style,
    layout=common_layout
)

nursery_w = widgets.Dropdown(
    options=[("Yes", "yes"), ("No", "no")],
    value="yes",
    description="Attended Nursery School:",
    style=common_style,
    layout=common_layout
)

higher_w = widgets.Dropdown(
    options=[("Yes", "yes"), ("No", "no")],
    value="yes",
    description="Wants Higher Education:",
    style=common_style,
    layout=common_layout
)

internet_w = widgets.Dropdown(
    options=[("Yes", "yes"), ("No", "no")],
    value="yes",
    description="Internet Access at Home:",
    style=common_style,
    layout=common_layout
)

romantic_w = widgets.Dropdown(
    options=[("Yes", "yes"), ("No", "no")],
    value="no",
    description="In a Romantic Relationship:",
    style=common_style,
    layout=common_layout
)

famrel_w = widgets.IntSlider(
    value=4, min=1, max=5, step=1,
    description="Family Relationship Quality (1–5):",
    style=common_style,
    layout=common_layout
)

freetime_w = widgets.IntSlider(
    value=3, min=1, max=5, step=1,
    description="Free Time After School (1–5):",
    style=common_style,
    layout=common_layout
)

goout_w = widgets.IntSlider(
    value=3, min=1, max=5, step=1,
    description="Going Out with Friends (1–5):",
    style=common_style,
    layout=common_layout
)

dalc_w = widgets.IntSlider(
    value=1, min=1, max=5, step=1,
    description="Weekday Alcohol Consumption (1–5):",
    style=common_style,
    layout=common_layout
)

walc_w = widgets.IntSlider(
    value=2, min=1, max=5, step=1,
    description="Weekend Alcohol Consumption (1–5):",
    style=common_style,
    layout=common_layout
)

health_w = widgets.IntSlider(
    value=4, min=1, max=5, step=1,
    description="Self-Reported Health Status (1–5):",
    style=common_style,
    layout=common_layout
)

target_w = widgets.FloatSlider(
    value=15, min=0, max=20, step=0.5,
    description="Target Final Grade (0–20):",
    style=common_style,
    layout=common_layout
)

run_btn = widgets.Button(description="Predict + Recommend", button_style="success")
out = widgets.Output()

scale_help = widgets.HTML(
    value="""
    <div style="line-height:1.35; margin-top:6px; margin-bottom:10px">
      <b>Scale notes</b>
      <ul>
        <li><b>Weekly Study Time (1–4):</b> 1=&lt;2h, 2=2–5h, 3=5–10h, 4=&gt;10h per week</li>
        <li><b>Travel Time (1–4):</b> 1=&lt;15m, 2=15–30m, 3=30–60m, 4=&gt;60m</li>
        <li><b>1–5 ratings:</b> 1=Very low, 5=Very high</li>
      </ul>
    </div>
    """
)

# -----------------------------
# Build student dict (matches model)
# -----------------------------
def build_student_dict():
    return {
        "sex": sex_w.value,
        "age": age_w.value,
        "address": address_w.value,
        "famsize": famsize_w.value,
        "Pstatus": pstatus_w.value,
        "Medu": medu_w.value,
        "Fedu": fedu_w.value,
        "Mjob": mjob_w.value,
        "Fjob": fjob_w.value,
        "reason": reason_w.value,
        "guardian": guardian_w.value,
        "traveltime": traveltime_w.value,
        "studytime": studytime_w.value,
        "failures": failures_w.value,
        "schoolsup": schoolsup_w.value,
        "famsup": famsup_w.value,
        "paid": paid_w.value,
        "activities": activities_w.value,
        "nursery": nursery_w.value,
        "higher": higher_w.value,
        "internet": internet_w.value,
        "romantic": romantic_w.value,
        "famrel": famrel_w.value,
        "freetime": freetime_w.value,
        "goout": goout_w.value,
        "Dalc": dalc_w.value,
        "Walc": walc_w.value,
        "health": health_w.value,
        "absences": absences_w.value
    }

# -----------------------------
# Button action
# -----------------------------
def on_run_clicked(_):
    student = build_student_dict()
    pred = predict_grade(student)
    typical, suggestions = recommend_changes(student, target_w.value, raw_all)

    with out:
        clear_output(wait=True)
        display(Markdown("## 🎯 Prediction Result"))
        display(Markdown(f"**Predicted Final Grade:** `{pred}` / 20"))
        display(Markdown(f"**Target Final Grade:** `{target_w.value}` / 20"))
        display(Markdown("---"))

        display(Markdown("## 📊 Factors Explained (You vs Typical for Target)"))
        comparison_df = make_comparison_table(student, typical)
        display(comparison_df)

        display(Markdown("## 📈 Visual Comparison (You vs Typical)"))
        display(Markdown(f"```text\n{bar_visual(student, typical)}\n```"))

        display(Markdown("## 💡 Suggestions"))
        for s in suggestions:
            display(Markdown(f"- {s}"))

run_btn.on_click(on_run_clicked)

# -----------------------------
# Layout
# -----------------------------
left = VBox([
    widgets.HTML("<h3>Student Information</h3>"),
    sex_w, age_w, address_w, famsize_w, pstatus_w,
    guardian_w, reason_w,
    medu_w, fedu_w, mjob_w, fjob_w
])

right = VBox([
    widgets.HTML("<h3>Study, Support & Lifestyle</h3>"),
    scale_help,
    traveltime_w, studytime_w, failures_w, absences_w,
    schoolsup_w, famsup_w, paid_w, activities_w,
    nursery_w, higher_w, internet_w, romantic_w,
    famrel_w, freetime_w, goout_w, dalc_w, walc_w, health_w,
    widgets.HTML("<h3>Goal</h3>"),
    target_w,
    run_btn
])

display(HBox([left, right]))
display(out)

Output()